In [ ]:
import os
import sys

repo_root_path = os.path.abspath(os.path.join("..", "..", ".."))

if repo_root_path not in sys.path:
    sys.path.append(repo_root_path)


In [ ]:
from analyse.advertising.e01_rupture_detection.script import (
    RuptureDetector,
    print_summary,
    export_json,
    plot_results,
)
from dateutil import tz
from datetime import datetime
from analyse.advertising.tools.testimony_table.extract import get_testimony_data
from analyse.advertising.tools.segment_player.player_generator import (
    format_annotation,
    generate_player,
)

tz_paris = tz.gettz('Europe/Paris')
start_time = datetime(2025, 3, 4, 13, 00, 0, tzinfo=tz_paris)
end_time = datetime(2025, 3, 4, 14, 0, 0, tzinfo=tz_paris)

TESTIMONY_CHANNEL = "TF1"

# This file was saved with utc datetime:
INPUT_FILE = "/root/Workspace/quotaclimat/analyse/advertising/exports/tf1_2025-03-04_12-00-00_2025-03-04_13-00-00.mp3"
VIDEO_INPUT = (
    "https://vod.mediatree.fr/assets/tf1_2025-03-04T12-00-00Z_2025-03-04T13-00-00Z.mp4"
)

OUT_JSON = "segments.json"
OUT_PLOT = "rupture_analyse.png"

annotations = get_testimony_data(
    channel=TESTIMONY_CHANNEL,
    from_date=start_time,
    to_date=end_time,
)

detector = RuptureDetector(
    sensitivity=0.35,
    context_sec=1.0,  # durée analysée de chaque côté d'un point pour évaluer la rupture.
    max_ruptures=0,  # 0 pour pas de limite
    sr=22050,  # Fréquence d'échantillonnage de travail
    hop_length=512,  # Pas entre frames (~23ms à 22050Hz)
    n_mfcc=20,  # Nombre de coefficients MFCC
    novelty_smooth_sec=0.5,  # Lissage temporel de la courbe de nouveauté.
    min_segment_sec=2.0,  # Durée minimale d'un segment pour être considéré comme tel.
)

In [ ]:

segments, novelty, features, y = detector.run(INPUT_FILE)

print_summary(segments)
export_json(segments, OUT_JSON)

plot_results(
    y,
    detector.sr,
    novelty,
    segments,
    hop_length=detector.hop_length,
    output_path=OUT_PLOT,
)

generate_player(
    media_input=VIDEO_INPUT,
    segments=[s.to_dict() for s in segments],
    annotations=format_annotation(annotations, from_date=start_time),
    output_path="output.html",
    params=detector.get_params(),
)